# L07 — Manual Tracing and Performance Measure Computation

**Module**: M03 | **Chapters**: 4–5 | **Lectures**: L07–L08

## Learning Objectives
By the end of this notebook you will be able to:
1. Construct a complete event trace table for a single-server queue.
2. Compute Wq, W, L, Lq, and ρ from trace data using the area-under-curve method.
3. Verify Little's Law numerically.
4. Explain the simultaneous-event convention and why it matters.

---
> **Think → Trace → Code → Experiment → Interpret → Communicate**

This notebook implements the manual trace in Python — not SimPy. The goal is to see exactly what happens inside a simulation before trusting a library to do it for you.
---

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

## 1. The Trace Engine

We implement a minimal event loop that drives a single-server FCFS queue.
This is *exactly* what Algorithm 4.1 (Event Loop) in the textbook does.

In [ ]:
def trace_single_server(arrivals, service_times, verbose=True):
    """
    Manually trace a single-server FCFS queue.

    Parameters
    ----------
    arrivals      : list of arrival times (pre-sorted ascending)
    service_times : list of service times (one per customer, same order)

    Returns
    -------
    customer_df : per-customer records
    events_df   : full event log with system state after each event
    """
    n = len(arrivals)
    assert len(service_times) == n

    svc_starts = np.zeros(n)
    svc_ends   = np.zeros(n)
    server_free_at = 0.0

    for i in range(n):
        svc_starts[i] = max(arrivals[i], server_free_at)
        svc_ends[i]   = svc_starts[i] + service_times[i]
        server_free_at = svc_ends[i]

    # Build event list: (time, type, customer_id)
    # Simultaneous events: departure (-1) before arrival (+1)
    ev = []
    for i in range(n):
        ev.append((arrivals[i],  +1, i, 'A'))  # arrival
        ev.append((svc_ends[i],  -1, i, 'D'))  # departure

    # Sort: primary=time, secondary=type (D before A at same time)
    ev.sort(key=lambda x: (x[0], x[1]))  # -1 < +1 so D sorts before A

    # Walk events, track n(t) and accumulate area
    rows = []
    cur_n = 0
    prev_t = 0.0
    cumulative_area_L  = 0.0
    cumulative_area_Lq = 0.0
    busy_time = 0.0
    prev_in_service = False

    for (t, delta, cid, etype) in ev:
        dt = t - prev_t
        cumulative_area_L  += cur_n * dt
        cumulative_area_Lq += max(cur_n - 1, 0) * dt
        if prev_in_service:
            busy_time += dt

        cur_n += delta
        in_service = cur_n > 0
        queue_len = max(cur_n - 1, 0)

        wait = svc_starts[cid] - arrivals[cid] if etype == 'A' else None

        rows.append({
            'time':         t,
            'event':        etype,
            'customer':     cid + 1,
            'n_after':      cur_n,
            'queue_len':    queue_len,
            'server_busy':  in_service,
            'wait':         wait,
            'cum_area_L':   cumulative_area_L,
            'cum_area_Lq':  cumulative_area_Lq,
        })
        prev_t = t
        prev_in_service = in_service

    T = svc_ends[-1]

    customer_df = pd.DataFrame({
        'customer':    np.arange(1, n+1),
        'arrival':     arrivals,
        'svc_start':   svc_starts,
        'svc_end':     svc_ends,
        'wait_Wq':     svc_starts - arrivals,
        'sojourn_W':   svc_ends   - arrivals,
    })
    events_df = pd.DataFrame(rows)

    results = {
        'T': T, 'n': n,
        'W_q': (svc_starts - arrivals).mean(),
        'W':   (svc_ends   - arrivals).mean(),
        'L':   cumulative_area_L  / T,
        'L_q': cumulative_area_Lq / T,
        'rho': busy_time / T,
        'lam': n / T,
    }

    if verbose:
        print(f"\n{'─'*60}")
        print(f"  T={T:.1f}   n={n} customers served")
        print(f"  W_q = {results['W_q']:.4f}   (mean wait in queue)")
        print(f"  W   = {results['W']:.4f}   (mean sojourn)")
        print(f"  L   = {results['L']:.4f}   (mean # in system)")
        print(f"  L_q = {results['L_q']:.4f}   (mean # in queue)")
        print(f"  ρ   = {results['rho']:.4f}   (server utilisation)")
        print(f"  λ̂   = {results['lam']:.4f}   (effective throughput)")
        print(f"{'─'*60}")
        lam = results['lam']
        print(f"  Little's Law  L  = λ̂W  : {lam*results['W']:.4f}  vs L={results['L']:.4f}")
        print(f"  Little's Law  Lq = λ̂Wq : {lam*results['W_q']:.4f}  vs Lq={results['L_q']:.4f}")
        print(f"  Decomposition L  = Lq+ρ: {results['L_q']+results['rho']:.4f}  vs L={results['L']:.4f}")

    return customer_df, events_df, results

## 2. Run the HW-02 Trace

These are the same arrival times and service times as in HW-02.

In [ ]:
arrivals_hw02   = [3, 8, 10, 18, 19, 23, 29, 32, 34, 41]
service_hw02    = [4,  7,  3,  5,  6,  2,  8,  4,  3,  5]

cust_df, ev_df, res = trace_single_server(arrivals_hw02, service_hw02)

print("\nCustomer-level summary:")
print(cust_df.to_string(index=False, float_format='{:.1f}'.format))

In [ ]:
print("Full event trace:")
display_cols = ['time','event','customer','n_after','queue_len','server_busy','wait','cum_area_L']
print(ev_df[display_cols].to_string(index=False, float_format='{:.2f}'.format))

## 3. Visualise n(t) — The Step Function

In [ ]:
def plot_state_trajectory(arrivals, service_times, ev_df, res):
    """Plot n(t) with shaded area = L*T, and highlight the queue (Lq)."""
    T = res['T']

    # Build step function for n(t)
    times  = [0.0] + list(ev_df['time'])
    n_vals = [0]   + list(ev_df['n_after'])

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

    # Top: n(t) = total in system
    ax1.step(times + [T], n_vals + [0], where='post',
             color='steelblue', lw=1.5, label='n(t) — total in system')
    ax1.fill_between(times + [T], n_vals + [0], step='post',
                     alpha=0.2, color='steelblue')
    ax1.axhline(res['L'], color='steelblue', linestyle='--', lw=1,
                label=f"L = {res['L']:.3f} (time-average)")
    ax1.set_ylabel('n(t) — # in system')
    ax1.legend(fontsize=8)
    ax1.grid(True, alpha=0.3)
    ax1.set_title(f'Single-server queue trace  (T={T}, W_q={res["W_q"]:.2f}, ρ={res["rho"]:.3f})')

    # Bottom: nq(t) = in queue only
    nq_vals = [max(v-1, 0) for v in n_vals]
    ax2.step(times + [T], nq_vals + [0], where='post',
             color='tomato', lw=1.5, label='nq(t) — in queue (waiting)')
    ax2.fill_between(times + [T], nq_vals + [0], step='post',
                     alpha=0.2, color='tomato')
    ax2.axhline(res['L_q'], color='tomato', linestyle='--', lw=1,
                label=f"Lq = {res['L_q']:.3f} (time-average)")

    # Mark arrivals and departures
    arr_t = ev_df[ev_df['event']=='A']['time']
    dep_t = ev_df[ev_df['event']=='D']['time']
    ax2.axvline(x=0, lw=0)  # force xlim
    for t in arr_t:
        ax2.axvline(t, color='green', alpha=0.3, lw=0.8)
    for t in dep_t:
        ax2.axvline(t, color='red', alpha=0.3, lw=0.8)

    ax2.set_xlabel('Simulated time (min)')
    ax2.set_ylabel('nq(t) — # in queue')
    ax2.legend(fontsize=8)
    ax2.grid(True, alpha=0.3)

    green_patch = mpatches.Patch(color='green', alpha=0.4, label='Arrival')
    red_patch   = mpatches.Patch(color='red',   alpha=0.4, label='Departure')
    ax2.legend(handles=[green_patch, red_patch] + ax2.get_legend_handles_labels()[0],
               fontsize=8)

    plt.tight_layout()
    plt.show()

plot_state_trajectory(arrivals_hw02, service_hw02, ev_df, res)

## 4. Simultaneous Events: Departure-Before-Arrival Convention

In [ ]:
# There are three simultaneous events in the HW-02 trace:
# t=18: C3 departs AND C4 arrives
# t=23: C4 departs AND C6 arrives
# t=29: C5 departs AND C7 arrives

# Show how the order matters
sim_events = ev_df[ev_df['time'].isin([18, 23, 29])][['time','event','customer','n_after']]
print("Events at simultaneous times (correct order: D before A):")
print(sim_events.to_string(index=False))

print()
print("If we reversed the order (A before D at t=18):")
print("  C4 arrives first → n goes 1→2 (C4 joins queue)")
print("  C3 departs → n goes 2→1 (but C4 would have to wait extra ~0 time)")
print("  For infinitely small Δt: the difference vanishes, but convention must be consistent.")
print()
print("The D-before-A convention means: a departing customer frees the server")
print("BEFORE an arriving customer can seize it. This is the SimPy default.")

## 5. Little's Law from First Principles

We just verified numerically that L = λW and Lq = λWq. Here's the intuition:

In [ ]:
# Little's Law geometric interpretation
# The 'sojourn area' — each customer contributes a rectangle of height=1, width=W_i
# The sum of rectangles equals the area under n(t)

cust = cust_df.copy()
T = res['T']

print(f"Sum of sojourn times = {cust['sojourn_W'].sum():.1f}")
print(f"Area under n(t) curve = {res['L'] * T:.1f}")
print()
print("These are equal (sample-path proof of Little's Law):")
print("  Area under n(t) = Σ_i W_i   (each customer spends W_i time units in the system)")
print(f"  ∴ L = (1/T) × Σ W_i = (n/T) × (1/n) Σ W_i = λ̂ × W̄")
print()
print(f"  λ̂ × W̄ = {res['lam']:.4f} × {res['W']:.4f} = {res['lam']*res['W']:.4f}")
print(f"  L      = {res['L']:.4f}")

## 6. What Changes with More Customers?

In [ ]:
# Generate a longer trace with random interarrival and service times
rng = np.random.default_rng(42)
lam_true, mu_true = 3.0, 4.0  # M/M/1 parameters
n_customers = 200

ia_times  = rng.exponential(1.0/lam_true, size=n_customers)
arrivals_long = np.cumsum(ia_times)
services_long = rng.exponential(1.0/mu_true, size=n_customers)

_, _, res_long = trace_single_server(arrivals_long, services_long, verbose=True)

# Compare to M/M/1 theory
rho = lam_true / mu_true
Wq_theory = lam_true / (mu_true * (mu_true - lam_true))
W_theory  = 1.0 / (mu_true - lam_true)
print(f"\nM/M/1 theory (λ={lam_true}, μ={mu_true}, ρ={rho:.2f}):")
print(f"  Wq = {Wq_theory:.4f}   W = {W_theory:.4f}   L = {rho/(1-rho):.4f}")

---
## Try It Yourself

1. **Change the load**: Repeat the 200-customer trace with λ=3.8 (ρ=0.95). How do Wq, L, and the simulation run time change compared to ρ=0.75? Does the system still come close to theory with 200 customers?

2. **Modify the trace engine**: Add a `finite_buffer` parameter: if a customer arrives when `n >= K`, the customer is turned away (blocked). Count the blocking probability and compare to the M/M/1/K formula from Chapter 7.

3. **Two servers**: Extend `trace_single_server` to support 2 servers. Each server has its own free-time tracker (`server_free = [0.0, 0.0]`). An arriving customer picks the server that becomes free soonest. Verify that the mean wait is lower than for 1 server at the same total capacity.

4. **Manual verification of HW-02**: Use the `trace_single_server` function with the HW-02 data to check your hand-computed answers. If any differ, identify which event you mis-processed.